# FRED Economic Indicators for Austin Real Estate Market

This notebook fetches economic data from the Federal Reserve Economic Data (FRED) API and generates 8 charts:

1. **Austin Employment - Office Sectors** (Professional, Financial, Government, Tech)
2. **Austin Employment - Industrial** (Trade & Transportation)
3. **Austin Employment - Retail** (Leisure & Hospitality)
4. **Austin vs National Tech Employment Growth** (indexed comparison)
5. **Austin Population Growth**
6. **Austin vs National Wage Growth** (indexed comparison)
7. **Interest Rates** (10-Year Treasury vs 30-Year Mortgage)
8. **Inflation** (Core CPI vs Rent CPI, indexed comparison)

All charts use Aquila brand styling and are saved to the `charts/` directory.

## Setup and Helper Functions

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from aquila_graphing_tools import aquila_styled_line_chart, AQUILA_COLORS, AQUILA_FONT

# Load environment variables
load_dotenv('aquila_graph.env')
fred_api_key = os.getenv('FRED_API_KEY')

if not fred_api_key:
    raise ValueError("FRED_API_KEY not found in aquila_graph.env")

print("✓ Environment loaded successfully")

In [ ]:
def fetch_fred_series(series_id, series_name=None):
    """
    Fetch FRED data and return as DataFrame with date and value columns.
    
    Parameters
    ----------
    series_id : str
        FRED series identifier
    series_name : str, optional
        Name for the value column (defaults to series_id)
    
    Returns
    -------
    pd.DataFrame
        DataFrame with 'date' and series_name columns
    """
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": fred_api_key,
        "file_type": "json"
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        observations = data.get("observations", [])
        
        if not observations:
            print(f"Warning: No data returned for series {series_id}")
            return pd.DataFrame()
        
        df = pd.DataFrame(observations)
        df['date'] = pd.to_datetime(df['date'])
        
        # Convert value to numeric, handling '.' as NaN
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        
        # Rename value column to series name
        column_name = series_name if series_name else series_id
        df = df[['date', 'value']].rename(columns={'value': column_name})
        
        # Drop NaN values
        df = df.dropna()
        
        print(f"✓ Fetched {len(df)} observations for {series_id} ({column_name})")
        return df
        
    except Exception as e:
        print(f"Error fetching series {series_id}: {str(e)}")
        return pd.DataFrame()

print("✓ Helper function defined")

## Chart 1: Austin Employment - Office Sectors

Tracks employment in office-related sectors:
- Professional & Business Services
- Financial Activities
- Government
- Information (Tech)

In [ ]:
# Fetch employment data for office sectors
df_prof = fetch_fred_series('AUST448PBSV', 'Professional & Business Services')
df_fire = fetch_fred_series('AUST448FIRE', 'Financial Activities')
df_govt = fetch_fred_series('AUST448GOVT', 'Government')
df_info = fetch_fred_series('AUST448INFO', 'Information (Tech)')

# Merge all series on date
df_office = df_prof.merge(df_fire, on='date', how='outer') \
                   .merge(df_govt, on='date', how='outer') \
                   .merge(df_info, on='date', how='outer')

# Convert to long format for plotting
df_office_long = df_office.melt(id_vars=['date'], 
                                 var_name='Sector', 
                                 value_name='Employment (thousands)')

# Create chart
fig = aquila_styled_line_chart(
    df_office_long,
    x='date',
    y='Employment (thousands)',
    color='Sector',
    title='Austin Employment - Office Sectors',
    height=800
)

fig.update_yaxes(rangemode='tozero')

# Save chart
fig.write_html('charts/austin_employment_office_sectors.html')
print("✓ Chart saved: austin_employment_office_sectors.html")

fig.show()

## Chart 2: Austin Employment - Industrial Sector

Tracks employment in Trade, Transportation & Utilities (industrial-related employment)

In [ ]:
# Fetch industrial employment data
df_industrial = fetch_fred_series('AUST448TRAD', 'Trade, Transportation & Utilities')

# Create chart
fig = aquila_styled_line_chart(
    df_industrial,
    x='date',
    y='Trade, Transportation & Utilities',
    title='Austin Employment - Industrial Sector',
    height=800
)

fig.update_yaxes(rangemode='tozero', title='Employment (thousands)')

# Save chart
fig.write_html('charts/austin_employment_industrial.html')
print("✓ Chart saved: austin_employment_industrial.html")

fig.show()

## Chart 3: Austin Employment - Retail Sector

Tracks employment in Leisure & Hospitality (retail-related employment)

In [ ]:
# Fetch retail employment data
df_retail = fetch_fred_series('AUST448LEIH', 'Leisure & Hospitality')

# Create chart
fig = aquila_styled_line_chart(
    df_retail,
    x='date',
    y='Leisure & Hospitality',
    title='Austin Employment - Retail Sector',
    height=800
)

fig.update_yaxes(rangemode='tozero', title='Employment (thousands)')

# Save chart
fig.write_html('charts/austin_employment_retail.html')
print("✓ Chart saved: austin_employment_retail.html")

fig.show()

## Chart 4: Austin vs National Tech Employment Growth

Compares tech employment growth between Austin and the nation, indexed to 100 at the earliest common date

In [ ]:
# Fetch tech employment data
df_austin_tech = fetch_fred_series('AUST448INFO', 'Austin Tech')
df_national_tech = fetch_fred_series('USINFO', 'National Tech')

# Merge on date
df_tech = df_austin_tech.merge(df_national_tech, on='date', how='inner')

# Find earliest common date and index both series to 100
if len(df_tech) > 0:
    base_austin = df_tech['Austin Tech'].iloc[0]
    base_national = df_tech['National Tech'].iloc[0]
    
    df_tech['Austin Tech (Index)'] = (df_tech['Austin Tech'] / base_austin) * 100
    df_tech['National Tech (Index)'] = (df_tech['National Tech'] / base_national) * 100
    
    # Convert to long format
    df_tech_long = df_tech[['date', 'Austin Tech (Index)', 'National Tech (Index)']].melt(
        id_vars=['date'],
        var_name='Region',
        value_name='Employment Index (Base 100)'
    )
    
    # Create chart
    fig = aquila_styled_line_chart(
        df_tech_long,
        x='date',
        y='Employment Index (Base 100)',
        color='Region',
        title='Austin vs National Tech Employment Growth',
        height=800
    )
    
    fig.update_yaxes(rangemode='tozero')
    
    # Save chart
    fig.write_html('charts/austin_vs_national_tech_employment.html')
    print("✓ Chart saved: austin_vs_national_tech_employment.html")
    
    fig.show()
else:
    print("Warning: No overlapping data for tech employment comparison")

## Chart 5: Austin Population Growth

Tracks Austin metro population over time. Auto-selects the most recent series between two available options.

In [ ]:
# Fetch both population series
df_pop1 = fetch_fred_series('AUST448POP', 'Population')
df_pop2 = fetch_fred_series('METRO12420MM428SCEN', 'Population')

# Select the series with the most recent data
if not df_pop1.empty and not df_pop2.empty:
    max_date1 = df_pop1['date'].max()
    max_date2 = df_pop2['date'].max()
    
    if max_date1 >= max_date2:
        df_population = df_pop1
        selected_series = 'AUST448POP'
    else:
        df_population = df_pop2
        selected_series = 'METRO12420MM428SCEN'
    
    print(f"Selected series {selected_series} with data through {df_population['date'].max().strftime('%Y-%m-%d')}")
    
elif not df_pop1.empty:
    df_population = df_pop1
    selected_series = 'AUST448POP'
    print(f"Using AUST448POP (only available series)")
    
elif not df_pop2.empty:
    df_population = df_pop2
    selected_series = 'METRO12420MM428SCEN'
    print(f"Using METRO12420MM428SCEN (only available series)")
    
else:
    print("Error: No population data available")
    df_population = pd.DataFrame()

# Create chart if data is available
if not df_population.empty:
    fig = aquila_styled_line_chart(
        df_population,
        x='date',
        y='Population',
        title='Austin Metro Population Growth',
        height=800
    )
    
    fig.update_yaxes(rangemode='tozero', title='Population (thousands)')
    
    # Save chart
    fig.write_html('charts/austin_population_growth.html')
    print("✓ Chart saved: austin_population_growth.html")
    
    fig.show()

## Chart 6: Austin vs National Wage Growth

Compares wage growth between Austin and the nation, indexed to 100 at the earliest common date.
National data (hourly) is converted to weekly by multiplying by 40 for comparison.

In [ ]:
# Fetch wage data
df_austin_wage = fetch_fred_series('AUST448AVGW', 'Austin Weekly Wage')
df_national_wage = fetch_fred_series('CES0500000003', 'National Hourly Wage')

# Convert national hourly to weekly (×40 hours)
df_national_wage['National Weekly Wage'] = df_national_wage['National Hourly Wage'] * 40
df_national_wage = df_national_wage[['date', 'National Weekly Wage']]

# Merge on date
df_wage = df_austin_wage.merge(df_national_wage, on='date', how='inner')

# Index both series to 100 at earliest common date
if len(df_wage) > 0:
    base_austin = df_wage['Austin Weekly Wage'].iloc[0]
    base_national = df_wage['National Weekly Wage'].iloc[0]
    
    df_wage['Austin Wage (Index)'] = (df_wage['Austin Weekly Wage'] / base_austin) * 100
    df_wage['National Wage (Index)'] = (df_wage['National Weekly Wage'] / base_national) * 100
    
    # Convert to long format
    df_wage_long = df_wage[['date', 'Austin Wage (Index)', 'National Wage (Index)']].melt(
        id_vars=['date'],
        var_name='Region',
        value_name='Wage Index (Base 100)'
    )
    
    # Create chart
    fig = aquila_styled_line_chart(
        df_wage_long,
        x='date',
        y='Wage Index (Base 100)',
        color='Region',
        title='Austin vs National Wage Growth',
        height=800
    )
    
    fig.update_yaxes(rangemode='tozero')
    
    # Save chart
    fig.write_html('charts/austin_vs_national_wage_growth.html')
    print("✓ Chart saved: austin_vs_national_wage_growth.html")
    
    fig.show()
else:
    print("Warning: No overlapping data for wage comparison")

## Chart 7: Interest Rates - 10-Year Treasury vs 30-Year Mortgage

Tracks two key interest rates that impact commercial real estate financing

In [ ]:
# Fetch interest rate data
df_treasury = fetch_fred_series('DGS10', '10-Year Treasury')
df_mortgage = fetch_fred_series('MORTGAGE30US', '30-Year Mortgage')

# Merge on date
df_rates = df_treasury.merge(df_mortgage, on='date', how='outer')

# Convert to long format
df_rates_long = df_rates.melt(
    id_vars=['date'],
    var_name='Rate Type',
    value_name='Interest Rate (%)'
)

# Create chart
fig = aquila_styled_line_chart(
    df_rates_long,
    x='date',
    y='Interest Rate (%)',
    color='Rate Type',
    title='Interest Rates - Treasury & Mortgage',
    height=800
)

fig.update_yaxes(rangemode='tozero')

# Save chart
fig.write_html('charts/interest_rates_treasury_mortgage.html')
print("✓ Chart saved: interest_rates_treasury_mortgage.html")

fig.show()

## Chart 8: Inflation - Core CPI vs Rent CPI

Compares general inflation (Core CPI) with rent inflation, indexed to 100 at earliest common date

In [ ]:
# Fetch inflation data
df_core_cpi = fetch_fred_series('CPILFESL', 'Core CPI')
df_rent_cpi = fetch_fred_series('CUUR0000SEHC', 'Rent CPI')

# Merge on date
df_inflation = df_core_cpi.merge(df_rent_cpi, on='date', how='inner')

# Index both series to 100 at earliest common date
if len(df_inflation) > 0:
    base_core = df_inflation['Core CPI'].iloc[0]
    base_rent = df_inflation['Rent CPI'].iloc[0]
    
    df_inflation['Core CPI (Index)'] = (df_inflation['Core CPI'] / base_core) * 100
    df_inflation['Rent CPI (Index)'] = (df_inflation['Rent CPI'] / base_rent) * 100
    
    # Convert to long format
    df_inflation_long = df_inflation[['date', 'Core CPI (Index)', 'Rent CPI (Index)']].melt(
        id_vars=['date'],
        var_name='Inflation Type',
        value_name='CPI Index (Base 100)'
    )
    
    # Create chart
    fig = aquila_styled_line_chart(
        df_inflation_long,
        x='date',
        y='CPI Index (Base 100)',
        color='Inflation Type',
        title='Inflation - Core CPI vs Rent CPI',
        height=800
    )
    
    fig.update_yaxes(rangemode='tozero')
    
    # Save chart
    fig.write_html('charts/inflation_core_vs_rent_cpi.html')
    print("✓ Chart saved: inflation_core_vs_rent_cpi.html")
    
    fig.show()
else:
    print("Warning: No overlapping data for inflation comparison")

## Summary

All 8 charts have been generated and saved to the `charts/` directory:

1. ✓ `austin_employment_office_sectors.html`
2. ✓ `austin_employment_industrial.html`
3. ✓ `austin_employment_retail.html`
4. ✓ `austin_vs_national_tech_employment.html`
5. ✓ `austin_population_growth.html`
6. ✓ `austin_vs_national_wage_growth.html`
7. ✓ `interest_rates_treasury_mortgage.html`
8. ✓ `inflation_core_vs_rent_cpi.html`

These charts are ready to be published to GitHub Pages and linked in README.md.